# 1.1 Load the Master Plan 2019 subzones

Every later step is scoped to URA's Master Plan 2019 subzones: HDB blocks are clustered within a
subzone (1_03), SingStat publishes child counts per subzone (1_04), and the final analysis table is
one row per subzone (1_08). This notebook loads the subzone polygons once, projects them to SVY21
(EPSG:3414, metres, so areas and distances downstream are in metres) and stores them as layer
`subzones` of the shared GeoPackage `output/hdb_blocks.gpkg`. Original script:
`scripts/load_subzones.py` (Minh).

**Inputs**
- `data/MasterPlanSubZones_2019.geojson` — Master Plan 2019 Subzone Boundary (No Sea),
  data.gov.sg `d_8594ae9ff96d0c708bc2af633048edfb` (version 2025-12-03).

**Outputs**
- `output/hdb_blocks.gpkg`, layer `subzones` — 332 subzones, EPSG:3414, with `area_km2`;
  read by 1_03 (cluster within subzone), 1_08 (subzone table) and the ESDA notebooks.

In [1]:
import sys
from pathlib import Path

import pyogrio

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

SRC = DATA / "MasterPlanSubZones_2019.geojson"
OUT_GPKG = OUT / "hdb_blocks.gpkg"   # shared GeoPackage; each notebook writes its own layer(s)
OUT_LAYER = "subzones"
CRS_OUT = "EPSG:3414"                # SVY21 / Singapore TM, metres

# Columns kept from the data.gov.sg file (it also carries INC_CRC / FMEL_UPD_D bookkeeping fields).
KEEP = ["SUBZONE_NO", "SUBZONE_N", "SUBZONE_C", "CA_IND", "PLN_AREA_N", "PLN_AREA_C",
        "REGION_N", "REGION_C", "geometry"]

## 1. Load and project the subzones

332 subzones in 55 planning areas and 5 regions. `SUBZONE_C` is the join key used everywhere
downstream, so it is checked for uniqueness here rather than discovered later as duplicated rows.

In [2]:
# 1. Read the GeoJSON (WGS 84), keep the identifying columns, project to metres.
subzones = pyogrio.read_dataframe(SRC)[KEEP].to_crs(CRS_OUT)

# 2. Area in km2 from the projected geometry; 1_08 divides centre and child counts by it.
subzones["area_km2"] = subzones.geometry.area / 1e6

# 3. The code must identify one subzone, otherwise every later join would duplicate rows.
assert subzones["SUBZONE_C"].is_unique, "SUBZONE_C must be unique"

print(f"Subzones: {len(subzones)}; planning areas: {subzones['PLN_AREA_N'].nunique()}; "
      f"regions: {subzones['REGION_N'].nunique()}; total area {subzones['area_km2'].sum():.1f} km2")
print(f"Largest: {subzones.loc[subzones['area_km2'].idxmax(), 'SUBZONE_N']} "
      f"({subzones['area_km2'].max():.1f} km2); median {subzones['area_km2'].median():.2f} km2")
subzones.drop(columns="geometry").head()

Subzones: 332; planning areas: 55; regions: 5; total area 784.8 km2
Largest: MURAI (68.6 km2); median 1.19 km2


,SUBZONE_NO,SUBZONE_N,SUBZONE_C,CA_IND,PLN_AREA_N,PLN_AREA_C,REGION_N,REGION_C,area_km2
0,12,DEPOT ROAD,BMSZ12,N,BUKIT MERAH,BM,CENTRAL REGION,CR,0.442298
1,2,BUKIT MERAH,BMSZ02,N,BUKIT MERAH,BM,CENTRAL REGION,CR,0.411723
2,3,CHINATOWN,OTSZ03,Y,OUTRAM,OT,CENTRAL REGION,CR,0.587223
3,4,PHILLIP,DTSZ04,Y,DOWNTOWN CORE,DT,CENTRAL REGION,CR,0.039438
4,5,RAFFLES PLACE,DTSZ05,Y,DOWNTOWN CORE,DT,CENTRAL REGION,CR,0.188767


## 2. Write the layer

In [3]:
# 1. Writing a layer into an existing GeoPackage replaces that layer only; other layers are untouched.
subzones.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
info = pyogrio.read_info(OUT_GPKG, layer=OUT_LAYER)
print(f"Wrote layer '{OUT_LAYER}' to {OUT_GPKG.relative_to(ROOT)}: {info['features']} features, {info['crs']}")

Wrote layer 'subzones' to output/hdb_blocks.gpkg: 332 features, EPSG:3414
